The goal of this notebook is to calculate ETO, ET0_rad, ET0_adv, and VPD for MIROC6 ssp126

In [1]:
import numpy as np
import xarray as xr
import cmip6_archive as ca
from datetime import datetime, timezone

def kelvin_to_celsius(da: xr.DataArray) -> xr.DataArray:
    """Convert temperature DataArray from K to °C if needed."""
    if da.attrs.get("units") == "K":
        da.attrs['units'] = "C"
        return da - 273.15
    else:
        raise AttributeError(f"Temperature must be in Kelvin but units are {da.attrs.get("units")}")

def net_surface_radiation(hfls: xr.DataArray, hfss: xr.DataArray) -> xr.DataArray:
    """Rn = hfls + hfss (W/m²), converted to MJ/m²/day. """
    if hfls.attrs.get("units") == "W m-2" and hfss.attrs.get("units") == "W m-2":
        return (hfls + hfss) * 86400 / 1e6 
    else:
        raise ValueError(f"Inputs must be in W m-2 but units are {hfls.attrs.get("units")}, {hfss.attrs.get("units")}")

def saturation_vapor_pressure(tas: xr.DataArray) -> xr.DataArray:
    """es in kPa from air temperature in °C. tas must be in °C."""
    if tas.attrs.get("units") != "C":
        raise ValueError(f"Temperature must be in ºC but it is in {tas.attrs.get("units")}")
    return 0.6108 * np.exp((17.27 * tas) / (tas + 237.3))

def actual_vapor_pressure(hurs: xr.DataArray, es: xr.DataArray) -> xr.DataArray:
    """ea in kPa from relative humidity (%) and saturation vapor pressure (es).
    Source: Eq. 19 FAO56 Guidelines (https://www.fao.org/4/x0490e/x0490e07.htm)"""
    return (hurs / 100.0) * es

def slope_saturation_vapor_pressure_curve(tas: xr.DataArray) -> xr.DataArray:
    """delta in kPa/°C tas must be in °C."""
    if tas.attrs.get("units") != "C":
        raise ValueError(f"Temperature must be in ºC but it is in {tas.attrs.get("units")}")

    return ( 2503 * np.exp((17.27 * tas) / (tas + 237.3)) ) / ((tas + 237.3) ** 2)

def psychrometric_constant(ps: xr.DataArray) -> xr.DataArray:
    """gamma in kPa/°C from surface pressure in Pa."""
    if ps.attrs.get("units") != "Pa":
        raise ValueError(f"Pressure must be in Pa but it is in {ps.attrs.get("units")}")
    return 0.000665 * (ps / 1000)

def wind_speed_2m(sfcWind: xr.DataArray, height: float = 10.0) -> xr.DataArray:
    """Rescale wind from measurement height to 2 m using FAO-56 log profile."""
    if sfcWind.attrs.get("units") != "m s-1":
        raise ValueError(f"Temperature must be in m s-1 but it is in {sfcWind.attrs.get("units")}")
    return sfcWind  * 4.87 / np.log(67.8 * height - 5.42)

def assign_ds_attrs(parent_ds_attrs: dict) -> dict:
    """
    Add dataset-level attributes: history, authors, source_id, variant_label, and experiment_id
    #TODO decide which attributes from parent dataset to keep
    """
    timestamp = (
                datetime.now(timezone.utc)
                .replace(microsecond=0)
                .isoformat()
                .replace("+00:00", "Z")
            )
    attrs = {
        "creation_date":    timestamp,
        "history":          f"{timestamp}: Created from data at National Taiwan University using ET0-test.ipynb",
        "authors":          "Marina Velasco-Barriuso (UPF)",
        "source_id":        parent_ds_attrs["source_id"],
        "variant_label":    parent_ds_attrs["variant_label"],
        "experiment_id":    parent_ds_attrs["experiment_id"],
    }

    return attrs

def penman_monteith(
    tas:      xr.DataArray,
    ps:       xr.DataArray,
    hfls:     xr.DataArray,
    hfss:     xr.DataArray,
    sfcWind:  xr.DataArray,
    hurs:     xr.DataArray,
    parent_ds_attrs: dict,
    height:   float = 10.0,
) -> xr.Dataset:
    """
    FAO-56 Penman-Monteith ET0 (mm/day). 
    Returns a Dataset with ET0, ET0_rad, ET0_adv, and VPD.
    """
    tas_c = kelvin_to_celsius(tas)

    # Calculate necessary variables
    # These functions convert units if needed
    Rn    = net_surface_radiation(hfls, hfss)
    delta = slope_saturation_vapor_pressure_curve(tas_c)
    gamma = psychrometric_constant(ps)
    U2    = wind_speed_2m(sfcWind, height=height)
    es  = saturation_vapor_pressure(tas_c)
    ea    = actual_vapor_pressure(hurs, es)

    # Calculate Vapor Pressure Deficit
    VPD   = es - ea

    # Calculate radiative and advective components separately
    denominator   = delta + gamma * (1 + 0.34 * U2)            # Common to both terms
    ET0_rad = (0.408 * delta * Rn) / denominator              # Assumes G ≈ 0
    ET0_adv = (gamma * 900 * U2 * VPD / (tas_c + 273)) / denominator

    # Calculate potential evapotranspiration as the sum of both terms
    ET0     = ET0_rad + ET0_adv

    # Assign variable-level attributes
    ET0 = ET0.assign_attrs(units="mm day-1", long_name="FAO-56 Penman-Monteith reference evapotranspiration")
    ET0_rad = ET0_rad.assign_attrs(units="mm day-1", long_name="Radiative component of ET0")
    ET0_adv = ET0_adv.assign_attrs(units="mm day-1", long_name="Advective component of ET0")
    VPD = VPD.assign_attrs(units="kPa",     long_name="Vapor pressure deficit")

    # Combine the 4 variables in a single Dataset
    ds = xr.Dataset({
            "ET0":     ET0,
            "ET0_rad": ET0_rad,
            "ET0_adv": ET0_adv,
            "VPD":     VPD,
        })

    # Assign Dataset-level attributes: history, author, parent info, etc.
    ds.attrs = assign_ds_attrs(parent_ds_attrs=parent_ds_attrs)

    return ds


In [9]:
# --- Test on a single GCM/experiment ---
archive = ca.CMIP6LocalArchive(root="/work10/archive/CMIP6/CMIP-SSPs/")

gcm, exp = "MIROC6", "ssp126"

tas_ds     = archive.get_variable_dataset(gcm, exp, "tas")

# Read in data as Xarray DataArrays
tas     = archive.get_variable_dataset(gcm, exp, "tas")["tas"]
ps      = archive.get_variable_dataset(gcm, exp, "ps")["ps"]
hfls    = archive.get_variable_dataset(gcm, exp, "hfls")["hfls"]
hfss    = archive.get_variable_dataset(gcm, exp, "hfss")["hfss"]
sfcWind = archive.get_variable_dataset(gcm, exp, "sfcWind")["sfcWind"]
hurs    = archive.get_variable_dataset(gcm, exp, "hurs")["hurs"]

## Testing individual functions

In [ ]:
test_period = slice('2015-01', '2015-12')
hfls_test = hfls.sel(time=test_period)
hfss_test = hfss.sel(time=test_period)
rnet = net_surface_radiation(hfls_test, hfss_test)
rnet

## Single GCM / experiment test

In [ ]:
ds = penman_monteith(tas, ps, hfls, hfss, sfcWind, hurs, parent_ds_attrs=tas_ds.attrs)
print(ds)

# Export each variable in a separate file
for var in ds.data_vars:
    out = ds[[var]]          # Dataset containing only one variable
    path = f"/work/home/H.mvelasco/SSPs/daily-ET0/test-result/{var}_{gcm}_{exp}_2015-2100.nc"
    out.to_netcdf(path)

In [24]:
import xarray as xr
folder = '/work10/archive/CMIP6/CMIP-SSPs/CMIP6/ScenarioMIP/DKRZ/MPI-ESM1-2-HR/ssp126/r1i1p1f1/day/tas/gn/v20190710/'
filename = 'tas_day_MPI-ESM1-2-HR_ssp126_r1i1p1f1_gn_20150101-20191231.nc'
ET0 = xr.open_dataset(folder+filename, chunks="auto")

#ET0.VPD.mean(dim='time').plot()
start = ET0["time"].dt.strftime("%Y%m%d").values[0]
end = ET0["time"].dt.strftime("%Y%m%d").values[-1]
print(start, end)

20150101 20191231


In [2]:
import xarray as xr
path = '/work/home/H.mvelasco/SSPs/daily-ET0/test-result/VPD_MIROC6_ssp126.nc'
ET0 = xr.open_dataset(path, chunks="auto")

#ET0.VPD.mean(dim='time').plot()
ET0.time

<xarray.DataArray 'time' (time: 31411)> Size: 251kB
array(['2015-01-01T12:00:00.000000000', '2015-01-02T12:00:00.000000000',
       '2015-01-03T12:00:00.000000000', ..., '2100-12-29T12:00:00.000000000',
       '2100-12-30T12:00:00.000000000', '2100-12-31T12:00:00.000000000'],
      shape=(31411,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 251kB 2015-01-01T12:00:00 ... 2100-12-31T1...
    height   float64 8B ...
Attributes:
    bounds:         time_bnds
    axis:           T
    long_name:      time
    standard_name:  time